### Scraping Fixture Table From Fbref

In [1]:
from scraper_engine import FBRefScraper

# 1. Initialize the scraping engine
scraper = FBRefScraper()

# 2. Execute the data extraction pipeline
# This covers from "2019-2020" up to "2025-2026"
scraper.run_pipeline(start_year=2019, end_year=2025)

[*] Booting Undetected Browser Engine...


*** chromedriver to download = 147.0.7727.117 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/147.0.7727.117/win64/chromedriver-win64.zip ...
Download Complete!

Extracting ['chromedriver.exe'] from chromedriver-win64.zip ...
Unzip Complete!

The file [uc_driver.exe] was saved to:
C:\Users\GARETHE\Projects\Injury Viz\venv\Lib\site-packages\seleniumbase\drivers\
uc_driver.exe

Making [uc_driver.exe 147.0.7727.117] executable ...
[uc_driver.exe 147.0.7727.117] is now ready for use!


*** chromedriver to download = 147.0.7727.117 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/147.0.7727.117/win64/chromedriver-win64.zip ...
Download Complete!

Extracting ['chromedriver.exe'] from chromedriver-win64.zip ...
Unzip Complete!

The file [chromedriver.exe] was saved to:
C:\Users\GARETHE\Projects\Injury Viz\venv\Lib\site-packages\seleniumbase\drivers\
chromedriver.exe

Making [chromedriver.exe 14

### Concatenating the fixture tables

In [ ]:
import pandas as pd
import glob
import os
import re

# 1. Target the raw_data directory
folder_path = "raw data"
all_files = glob.glob(os.path.join(folder_path, "fixture_table_*.csv"))

data_list = []
print(f"[*] Found {len(all_files)} CSV files. Commencing merge and clean pipeline...")

for file in all_files:
    # Read the individual CSV
    df = pd.read_csv(file)
    
    # --- DATA CLEANING ---
    # 1. Clean Time format: "16:30 (17:30)" -> "16:30"
    if 'time' in df.columns:
        df['time'] = df['time'].astype(str).str.split(' ').str[0]
        
    # 2. Clean Opponent names: "es Athletic Club" -> "Athletic Club"
    # Regex looks for exactly two lowercase letters at the start of the string followed by a space
    if 'opponent' in df.columns:
        df['opponent'] = df['opponent'].astype(str).str.replace(r'^[a-z]{2}\s', '', regex=True)
    
    # --- METADATA INJECTION ---
    # Extract Team and Season from the filename so we don't lose context after merging
    filename = os.path.basename(file)
    match = re.search(r"fixture_table_(.*?)_(\d{4}-\d{4})", filename)
    
    if match:
        df['team_name'] = match.group(1).replace('_', ' ').title()
        df['season'] = match.group(2)
    
    data_list.append(df)

# 2. Row append all DataFrames together
master_df = pd.concat(data_list, ignore_index=True)

# 3. Final Ordering: Sort chronologically
master_df['date'] = pd.to_datetime(master_df['date'])
master_df = master_df.sort_values(by=['date', 'team_name']).reset_index(drop=True)

# 4. Export to disk
output_filename = "master_premier_league_fixtures.csv"
master_df.to_csv(output_filename, index=False)

print(f"[+] Pipeline Complete! Master dataset saved as '{output_filename}'.")
print(f"[*] Total Records: {len(master_df)}")
print(f"[*] Teams Processed: {master_df['team_name'].unique().tolist()}")

# Display the cleaned head of the master table
master_df.head()

### Scraping the absentee table from transfermarkt

In [1]:
from scraper_engine import TMAbsenceScraper

# 1. Initialize the Transfermarkt scraping engine
tm_scraper = TMAbsenceScraper()

# 2. Execute the extraction pipeline (2019/2020 through 2025/2026)
tm_scraper.run_pipeline(start_year=2019, end_year=2025)

[*] Booting Undetected Browser Engine for Transfermarkt...
[-] SKIPPED (Already Exists): Arsenal 2019-2020
[-] SKIPPED (Already Exists): Arsenal 2020-2021
[-] SKIPPED (Already Exists): Arsenal 2021-2022
[*] Fetching Absence Data: Arsenal | Season: 2022...
    -> Target Link: https://www.transfermarkt.co.za/arsenal-fc/ausfallzeiten/verein/11?reldata=GB1%262022
[+] SUCCESS: Saved to raw data\absence_table_arsenal_2022-2023.csv (42 players extracted)
--> Cooling down for 7.66s...

[-] SKIPPED (Already Exists): Arsenal 2023-2024
[-] SKIPPED (Already Exists): Arsenal 2024-2025
[-] SKIPPED (Already Exists): Arsenal 2025-2026
[*] Fetching Absence Data: Man_United | Season: 2019...
    -> Target Link: https://www.transfermarkt.co.za/manchester-united/ausfallzeiten/verein/985?reldata=GB1%262019
[+] SUCCESS: Saved to raw data\absence_table_man_united_2019-2020.csv (43 players extracted)
--> Cooling down for 7.31s...

[*] Fetching Absence Data: Man_United | Season: 2020...
    -> Target Link: htt

### Transforming and concatenating transfermrkt table

In [1]:
import pandas as pd
import glob
import os
import re

# 1. Target the raw data directory
folder_path = "raw data"
all_absence_files = glob.glob(os.path.join(folder_path, "absence_table_*.csv"))

data_list = []
print(f"[*] Found {len(all_absence_files)} Absence tables. Commencing Wide-to-Long transformation...")

for file in all_absence_files:
    # Read the individual Wide CSV
    df = pd.read_csv(file)
    
    # Extract Context from the filename
    filename = os.path.basename(file)
    match = re.search(r"absence_table_(.*?)_(\d{4}-\d{4})", filename)
    
    if not match:
        continue
        
    team_name = match.group(1).replace('_', ' ').title()
    season = match.group(2)
    
    # 2. Identify all the Gameweek columns dynamically
    gw_columns = [col for col in df.columns if str(col).startswith('GW_')]
    
    # 3. UNPIVOT THE DATA (Wide to Long Transformation)
    melted_df = pd.melt(
        df, 
        id_vars=['Player'],           # Columns to keep static
        value_vars=gw_columns,        # Columns to collapse into rows
        var_name='round',             # Name of the new 'variable' column
        value_name='player_status'    # Name of the new 'value' column
    )
    
    # 4. Standardize column names to match your schema requirements
    melted_df = melted_df.rename(columns={'Player': 'player_name'})
    melted_df['team_name'] = team_name
    melted_df['season'] = season
    
    # 5. Data Engineering Polish: Match the FBref naming convention
    # Converts "GW_1" into "Matchweek 1" so you can easily JOIN these tables later
    melted_df['round'] = melted_df['round'].str.replace('GW_', 'Matchweek ')
    
    data_list.append(melted_df)

# 6. Concatenate all transformed tables into one master DataFrame
master_absence_df = pd.concat(data_list, ignore_index=True)

# 7. Reorder columns for logical readability
master_absence_df = master_absence_df[['season', 'team_name', 'player_name', 'round', 'player_status']]

# 8. Sort the dataset so it flows chronologically by team and player
# We extract the integer from "Matchweek X" just for sorting purposes, then drop it
master_absence_df['sort_gw'] = master_absence_df['round'].str.extract(r'(\d+)').astype(int)
master_absence_df = master_absence_df.sort_values(by=['season', 'team_name', 'player_name', 'sort_gw']).drop(columns=['sort_gw'])
master_absence_df = master_absence_df.reset_index(drop=True)

# 9. Save the production-ready dataset
output_filename = "master_absence_long.csv"
output_folder = "processed data"

# Ensure the destination folder exists
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"[*] Created directory: {output_folder}")

# Safely join the folder path and filename
file_path = os.path.join(output_folder, output_filename)

# Save to the correct directory
master_absence_df.to_csv(file_path, index=False)
print(f"[+] Saved successfully to: {file_path}")
print(f"[*] Total Records Generated: {len(master_absence_df)}")

# Display the head to verify the new structure
master_absence_df.head(10)

[*] Found 42 Absence tables. Commencing Wide-to-Long transformation...
[+] Saved successfully to: processed data\master_absence_long.csv
[*] Total Records Generated: 64410


,season,team_name,player_name,round,player_status
0,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 1,Starting eleven
1,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 2,Starting eleven
2,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 3,Starting eleven
3,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 4,Starting eleven
4,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 5,Starting eleven
5,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 6,Starting eleven
6,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 7,On the bench
7,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 8,On the bench
8,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 9,Not included
9,2019-2020,Arsenal,Ainsley Maitland-Niles,Matchweek 10,Not included
